# DenseNet-169 evaluation — threshold sweep run

Run top to bottom in a **fresh** Colab runtime.

**Before running:** push the updated `scripts/evaluate.py` (with the threshold
sweep) to the `audit-fixes` branch. Cell 1 clones that branch, so if the new
version isn't pushed you'll get the old output with no sweep table.

You'll be asked to upload two files: `kaggle.json` and your `.pth` checkpoint.

## 1. Clone repo and install dependencies

In [ ]:
!git clone -b audit-fixes https://github.com/Pushkar0997/plant-disease-densenet169.git
%cd plant-disease-densenet169
!pip install -q -r requirements.txt

# Confirm the pushed evaluate.py actually has the sweep. If this prints
# MISSING, the updated script wasn't pushed — stop and push it first.
!grep -q "threshold_sweep" scripts/evaluate.py && echo "OK: threshold sweep present" || echo "MISSING: push updated evaluate.py first"

## 2. Kaggle auth
Upload `kaggle.json` when prompted (kaggle.com → Account → Create New API Token).

In [ ]:
import os
from pathlib import Path
from google.colab import files

KAGGLE_JSON_PATH = Path.home() / '.kaggle' / 'kaggle.json'
if not KAGGLE_JSON_PATH.exists():
    print("Upload your kaggle.json:")
    uploaded = files.upload()
    KAGGLE_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
    with open(KAGGLE_JSON_PATH, 'wb') as f:
        f.write(uploaded['kaggle.json'])
    os.chmod(KAGGLE_JSON_PATH, 0o600)
    print("Saved.")
else:
    print("kaggle.json already present in this runtime.")

## 3. Download the dataset
Must happen before evaluation — `evaluate.py` needs the images, not just the checkpoint.

In [ ]:
!pip install -q kaggle
!mkdir -p /content/plantvillage_data/raw
!kaggle datasets download -d emmarex/plantdisease -p /content/plantvillage_data/raw --unzip

### Sanity check — should list class-named folders

In [ ]:
!find /content/plantvillage_data/raw -maxdepth 3 -type d | head -20

## 4. Upload the checkpoint

`files.upload()` saves to the **current working directory**, which is
`/content/plant-disease-densenet169` because of the `%cd` above. That's why
the next cell uses a relative path.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select densenet169_plantvillage.pth

In [ ]:
!ls -la densenet169_plantvillage.pth

## 5. Run the evaluation
Prints the threshold sweep table at the end — that's what this run is for.

In [ ]:
!python scripts/evaluate.py \
    --data-root /content/plantvillage_data/raw \
    --checkpoint densenet169_plantvillage.pth \
    --class-mapping models/class_mapping.json \
    --output reports/plantvillage_eval.json

## 6. Download all three outputs
The `.predictions.csv` is new — it means any future re-analysis won't need another Colab run.

In [ ]:
from google.colab import files
files.download('reports/plantvillage_eval.json')
files.download('reports/plantvillage_eval.confusion.png')
files.download('reports/plantvillage_eval.predictions.csv')

---
## Optional: field photo evaluation (step 5)

Only run this once you have your own photographs uploaded to
`/content/field_photos/` with a `labels.csv`. PlantVillage images will not
work here — see the script docstring for why.

In [ ]:
# !python scripts/evaluate_field.py \
#     --images-dir /content/field_photos \
#     --labels-csv /content/field_photos/labels.csv \
#     --checkpoint densenet169_plantvillage.pth \
#     --output reports/field_eval.json